In [24]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("MyApp")
.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.1.1")
.master("local[*]")
.getOrCreate())

In [25]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


In [26]:
# First, delete topics if they exist
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})
try:
    admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
    print("Deleted old topics if they existed.")
except Exception as e:
    pass
# Create new topics
new_topics = [NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
              NewTopic(topic="movies", num_partitions=1, replication_factor=3),
              NewTopic(topic="tags", num_partitions=1, replication_factor=3)]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result() # The result itself is None
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Deleted old topics if they existed.
Topic ratings created
Topic movies created
Topic tags created


In [27]:
# Push the dataframes to kafka broker
# Push Ratings
df_ratings.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
    .option("topic", "ratings") \
    .save()
print("Pushed ratings successfully!")

# Push Movies
df_movies.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
    .option("topic", "movies") \
    .save()
print("Pushed movies successfully!")

# Push Tags
df_tags.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
    .option("topic", "tags") \
    .save()
print("Pushed tags successfully!")
    
# Check if data is in Kafka topics, however it's stored in binary format, and in "value" column
df_ratings_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
    .option("subscribe", "ratings") \
    .load()
    
df_ratings_kafka = df_ratings_kafka.selectExpr("CAST(value AS STRING)")
df_ratings_kafka.show(1)

26/04/09 22:18:27 WARN Sender: [Producer clientId=producer-2] Got error produce response with correlation id 1669 on topic-partition ratings-0, retrying (2147483646 attempts left). Error: NOT_LEADER_OR_FOLLOWER
26/04/09 22:18:27 WARN Sender: [Producer clientId=producer-2] Received invalid metadata error in produce request on partition ratings-0 due to org.apache.kafka.common.errors.NotLeaderOrFollowerException: For requests intended only for the leader, this error indicates that the broker is not the current leader. For requests intended for any replica, this error indicates that the broker is not a replica of the topic partition. Going to request metadata update now
26/04/09 22:18:27 WARN Sender: [Producer clientId=producer-2] Got error produce response with correlation id 1670 on topic-partition ratings-0, retrying (2147483646 attempts left). Error: NOT_LEADER_OR_FOLLOWER
26/04/09 22:18:27 WARN Sender: [Producer clientId=producer-2] Received invalid metadata error in produce request 

Pushed ratings successfully!
Pushed movies successfully!
Pushed tags successfully!
+--------------------+
|               value|
+--------------------+
|{"userId":1,"movi...|
+--------------------+
only showing top 1 row
